# Horizon Europe: business questions answered on the data model

**PUBLIC DATA**: CORDIS (Publications Office of the EU), Horizon Europe and Horizon 2020, downloaded 16 November 2026. Each section is a user story from `docs/analysis/REQUIREMENTS.md`; the SQL is read from `sql/` and runs on the DuckDB database built by `src/load.py`. Amounts are **maximum EU contributions** of signed grants (commitments, not payments). Descriptive figures only: they do not rank organisations or countries.

In [1]:
import duckdb, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
con = duckdb.connect(str(ROOT / 'data' / 'cordis.duckdb'), read_only=True)
pd.set_option('display.max_columns', 12)
def run(name, head=12):
    sql = (ROOT / 'sql' / f'{name}.sql').read_text(encoding='utf-8')
    print(sql)
    return con.execute(sql).df().head(head)

## Quality report of the load
Eighteen checks run after each load (`src/quality.py`). FAIL would stop the load; WARN are known properties of the source, explained in `docs/analysis/DATA_PROFILING.md`.

In [2]:
pd.read_csv(ROOT / 'data' / 'processed' / 'qa_report.csv')

,id,rule,offending_rows,status
0,Q01,Every project has exactly one coordinator,0,PASS
1,Q02,Every project has at least one participation,0,PASS
2,Q03,Raw rows = loaded rows (projects),0,PASS
3,Q04,Raw rows = loaded rows (participations),0,PASS
4,Q05,Control total: EU contribution loaded = EU con...,0,PASS
5,Q06,Unfunded roles (associated partners) carry no ...,16,WARN
6,Q07,Sum of participants' net contributions = proje...,325,WARN
7,Q08,Projects without a EuroSciVoc classification,7864,WARN
8,Q09,Participations without a country,347,WARN
9,Q10,Duplicate organisation candidates (same normal...,190,WARN


## US01 - Where is the budget committed? Funding by programme and pillar

In [3]:
run('us01_funding_by_pillar')

-- US01. EU contribution (maximum grant) and number of projects by programme and pillar.
SELECT programme_code, pillar_title,
       COUNT(*)                              AS projects,
       ROUND(SUM(ec_max_contribution) / 1e9, 2) AS eu_contribution_bn
FROM mart.dim_project
GROUP BY programme_code, pillar_title
ORDER BY programme_code, eu_contribution_bn DESC;



,programme_code,pillar_title,projects,eu_contribution_bn
0,H2020,Societal challenges,7109,26.42
1,H2020,Excellent science,20621,25.01
2,H2020,Industrial leadership,6587,13.80
3,H2020,Euratom,99,1.09
4,H2020,Spreading excellence and widening participation,488,1.01
5,H2020,Horizon 2020 Framework Programme,225,0.51
6,H2020,Science with and for society,260,0.49
7,HORIZON,Pillar II - Global Challenges and European Ind...,5484,36.15
8,HORIZON,Pillar I - Excellent Science,15949,19.68
9,HORIZON,Pillar III - Innovative Europe,1906,6.50


## US02 - Geographic spread: funding, participations and coordinations by country

In [4]:
run('us02_funding_by_country')

-- US02. Net EU contribution, participations and coordinations by country (Horizon Europe).
SELECT f.country_code, c.name AS country, c.is_eu27,
       COUNT(*)                                     AS participations,
       SUM(f.is_coordinator)                        AS coordinations,
       ROUND(SUM(f.net_ec_contribution) / 1e9, 3)   AS eu_contribution_bn
FROM mart.fact_participation f
LEFT JOIN country c ON c.country_code = f.country_code
WHERE f.programme_code = 'HORIZON'
GROUP BY f.country_code, c.name, c.is_eu27
ORDER BY eu_contribution_bn DESC;



,country_code,country,is_eu27,participations,coordinations,eu_contribution_bn
0,DE,Germany,True,16797,3166.0,10.258
1,FR,France,True,13711,2385.0,7.024
2,ES,Spain,True,16477,2973.0,6.745
3,IT,Italy,True,14172,2408.0,5.295
4,NL,Netherlands,True,9156,1873.0,5.265
5,BE,Belgium,True,8172,1307.0,4.534
6,GB,United Kingdom,False,8260,1399.0,2.475
7,GR,Greece,True,6537,784.0,2.398
8,SE,Sweden,True,4243,872.0,2.172
9,AT,Austria,True,4293,802.0,2.059


## US03 - Who receives the funding? Organisation types

In [5]:
run('us03_funding_by_org_type')

-- US03. Share of participations and of EU contribution by organisation type, both programmes.
SELECT f.programme_code, f.type_code, t.label,
       COUNT(*) AS participations,
       ROUND(SUM(f.net_ec_contribution) / 1e9, 2) AS eu_contribution_bn,
       ROUND(100.0 * SUM(f.net_ec_contribution) / SUM(SUM(f.net_ec_contribution)) OVER (PARTITION BY f.programme_code), 1) AS share_pct
FROM mart.fact_participation f
LEFT JOIN organisation_type t ON t.type_code = f.type_code
GROUP BY f.programme_code, f.type_code, t.label
ORDER BY f.programme_code, eu_contribution_bn DESC;



,programme_code,type_code,label,participations,eu_contribution_bn,share_pct
0,H2020,HES,Higher or secondary education establishment,59931,26.94,39.7
1,H2020,PRC,Private for-profit entity (excluding education),60095,18.87,27.8
2,H2020,REC,Research organisation,36881,17.12,25.2
3,H2020,OTH,Other,11919,2.83,4.2
4,H2020,PUB,Public body (excluding research and education),9762,2.06,3.0
5,H2020,None,None,475,0.09,0.1
6,HORIZON,HES,Higher or secondary education establishment,54157,24.43,37.3
7,HORIZON,REC,Research organisation,33041,17.74,27.1
8,HORIZON,PRC,Private for-profit entity (excluding education),44146,16.53,25.3
9,HORIZON,OTH,Other,12113,4.26,6.5


## US04 - Funding by scientific domain, without double counting (weighted bridge)

In [6]:
run('us04_funding_by_domain')

-- US04. EU contribution by top-level scientific domain (EuroSciVoc), allocated with the bridge weights
-- so that a project classified in two domains is not counted twice. Unclassified projects shown apart.
SELECT p.programme_code, COALESCE(b.domain_label, '(not classified)') AS domain,
       ROUND(SUM(p.ec_max_contribution * COALESCE(b.weight, 1)) / 1e9, 2) AS eu_contribution_bn,
       ROUND(SUM(COALESCE(b.weight, 1)), 0) AS projects_weighted
FROM mart.dim_project p
LEFT JOIN mart.bridge_project_domain b ON b.project_id = p.project_id
GROUP BY p.programme_code, COALESCE(b.domain_label, '(not classified)')
ORDER BY p.programme_code, eu_contribution_bn DESC;



,programme_code,domain,eu_contribution_bn,projects_weighted
0,H2020,natural sciences,22.45,12823.0
1,H2020,engineering and technology,16.43,7020.0
2,H2020,social sciences,10.22,5098.0
3,H2020,(not classified),7.95,3514.0
4,H2020,medical and health sciences,7.80,4526.0
5,H2020,agricultural sciences,1.82,953.0
6,H2020,humanities,1.65,1456.0
7,HORIZON,natural sciences,19.49,8566.0
8,HORIZON,(not classified),14.41,4350.0
9,HORIZON,engineering and technology,12.41,3681.0


## US05 - What does a consortium look like, by type of action?

In [7]:
run('us05_consortium_size')

-- US05. Consortium size (distinct organisations) and number of countries by type of action (Horizon Europe).
SELECT action_family,
       COUNT(*)                                                    AS projects,
       ROUND(AVG(consortium_size), 1)                              AS avg_members,
       QUANTILE_CONT(consortium_size, 0.5)                         AS median_members,
       ROUND(AVG(n_countries), 1)                                  AS avg_countries,
       ROUND(100.0 * AVG(CASE WHEN consortium_size = 1 THEN 1 ELSE 0 END), 1) AS single_beneficiary_pct
FROM mart.dim_project
WHERE programme_code = 'HORIZON'
GROUP BY action_family
ORDER BY projects DESC;



,action_family,projects,avg_members,median_members,avg_countries,single_beneficiary_pct
0,Marie Sklodowska-Curie,8814,3.9,2.0,2.4,49.9
1,ERC frontier research,6999,1.3,1.0,1.1,81.8
2,Research and innovation action,3552,15.2,13.0,8.2,0.1
3,Coordination and support action,1885,8.5,6.0,5.4,22.9
4,Innovation action,1547,19.4,17.0,8.4,0.0
5,EIC innovation,1531,3.1,1.0,2.3,57.5
6,Other,49,13.0,8.0,6.1,8.2
7,Co-fund,42,58.1,57.0,20.7,7.1


## US06 - Grant size and duration by type of action

In [8]:
run('us06_duration_and_size')

-- US06. Grant size and duration by type of action (Horizon Europe): median maximum EU contribution and months.
SELECT action_family,
       COUNT(*) AS projects,
       ROUND(QUANTILE_CONT(ec_max_contribution, 0.5) / 1e6, 2) AS median_grant_m,
       ROUND(SUM(ec_max_contribution) / 1e9, 2)                AS total_bn,
       QUANTILE_CONT(duration_months, 0.5)                     AS median_months
FROM mart.dim_project
WHERE programme_code = 'HORIZON'
GROUP BY action_family
ORDER BY total_bn DESC;



,action_family,projects,median_grant_m,total_bn,median_months
0,Research and innovation action,3552,4.97,21.65,42.0
1,ERC frontier research,6999,1.65,12.84,60.0
2,Innovation action,1547,6.48,12.68,47.0
3,Marie Sklodowska-Curie,8814,0.22,5.13,24.0
4,Coordination and support action,1885,1.50,4.24,36.0
5,EIC innovation,1531,2.50,3.99,32.0
6,Co-fund,42,35.00,2.68,78.0
7,Other,49,17.02,2.25,36.0


## US07 - Which countries work together?

In [9]:
run('us07_country_network')

-- US07. Cooperation between countries: number of Horizon Europe projects in which two countries take part together.
WITH pc AS (
    SELECT DISTINCT f.project_id, f.country_code
    FROM mart.fact_participation f
    WHERE f.programme_code = 'HORIZON' AND f.country_code IS NOT NULL
)
SELECT a.country_code AS country_a, b.country_code AS country_b, COUNT(*) AS shared_projects
FROM pc a JOIN pc b ON a.project_id = b.project_id AND a.country_code < b.country_code
GROUP BY a.country_code, b.country_code
ORDER BY shared_projects DESC;



,country_a,country_b,shared_projects
0,DE,ES,3679
1,DE,IT,3617
2,ES,IT,3425
3,DE,FR,3239
4,ES,FR,2913
5,DE,NL,2869
6,FR,IT,2856
7,BE,DE,2777
8,BE,ES,2619
9,BE,IT,2534


## US08 - Projects by start year, two programmes

In [10]:
run('us08_projects_by_start_year')

-- US08. Projects and EU contribution by start year, both programmes.
SELECT programme_code, start_year, COUNT(*) AS projects, ROUND(SUM(ec_max_contribution) / 1e9, 2) AS eu_contribution_bn
FROM mart.dim_project
GROUP BY programme_code, start_year
ORDER BY programme_code, start_year;



,programme_code,start_year,projects,eu_contribution_bn
0,H2020,2014,374,1.00
1,H2020,2015,4706,8.84
2,H2020,2016,4945,8.83
3,H2020,2017,4970,8.66
4,H2020,2018,5071,9.20
5,H2020,2019,5561,10.52
6,H2020,2020,4506,11.24
7,H2020,2021,4296,9.28
8,H2020,2022,934,0.77
9,H2020,2023,14,0.01


## US09 - Coordinations per participation, all actions and collaborative actions only

In [11]:
run('us09_coordination_share')

-- US09. Coordinator share by country (Horizon Europe, countries with 1,000+ participations).
-- Single-beneficiary grants (ERC, MSCA fellowships, EIC) make their only participant the coordinator, so the rate
-- is also shown for collaborative actions only (RIA, IA, CSA), where a consortium chooses its coordinator.
SELECT f.country_code, c.name AS country,
       COUNT(*) AS participations, SUM(f.is_coordinator) AS coordinations,
       ROUND(100.0 * SUM(f.is_coordinator) / COUNT(*), 1) AS coordination_rate_pct,
       ROUND(100.0 * SUM(f.is_coordinator) FILTER (WHERE s.action_family IN ('Research and innovation action', 'Innovation action',
             'Coordination and support action'))
             / COUNT(*) FILTER (WHERE s.action_family IN ('Research and innovation action', 'Innovation action',
             'Coordination and support action')), 1) AS collaborative_coordination_rate_pct
FROM mart.fact_participation f
JOIN project p ON p.project_id = f.project_id
JOIN funding_scheme

,country_code,country,participations,coordinations,coordination_rate_pct,collaborative_coordination_rate_pct
0,DE,Germany,16797,3166.0,18.8,6.8
1,ES,Spain,16477,2973.0,18.0,8.5
2,IT,Italy,14172,2408.0,17.0,6.8
3,FR,France,13711,2385.0,17.4,6.4
4,NL,Netherlands,9156,1873.0,20.5,7.8
5,GB,United Kingdom,8260,1399.0,16.9,3.0
6,BE,Belgium,8172,1307.0,16.0,7.3
7,GR,Greece,6537,784.0,12.0,10.5
8,AT,Austria,4293,802.0,18.7,7.5
9,SE,Sweden,4243,872.0,20.6,6.6


## US10 - A domain and all its sub-domains (recursive SQL)

In [12]:
run('us10_domain_drilldown')

-- US10. Drill-down: projects in a domain and all its sub-domains (recursive query on the EuroSciVoc tree),
-- here 'artificial intelligence', Horizon Europe. The SPARQL equivalent is semantic/queries/graph_01_domain_tree.rq.
WITH RECURSIVE sub AS (
    SELECT domain_code FROM science_domain WHERE label = 'artificial intelligence'
    UNION ALL
    SELECT d.domain_code FROM science_domain d JOIN sub ON d.parent_code = sub.domain_code
),
projects AS (
    SELECT DISTINCT psd.project_id FROM project_science_domain psd JOIN sub USING (domain_code)
)
SELECT (SELECT COUNT(*) FROM sub) AS domains_in_subtree,
       COUNT(*) AS projects,
       ROUND(SUM(p.ec_max_contribution) / 1e9, 2) AS eu_contribution_bn
FROM projects JOIN project p USING (project_id)
WHERE p.programme_code = 'HORIZON';



,domains_in_subtree,projects,eu_contribution_bn
0,21,1263,4.05


## US11 - Lineage: where does each row come from?

In [13]:
run('us11_lineage')

-- US11. Lineage: every project row points to the load batch (source file, download date, lines repaired).
SELECT b.batch_id, b.programme_code, b.source_url, b.downloaded_utc, b.loaded_utc, b.rows_read, b.lines_repaired,
       b.lines_rejected, COUNT(p.project_id) AS projects
FROM load_batch b LEFT JOIN project p USING (batch_id)
GROUP BY ALL ORDER BY b.batch_id;



,batch_id,programme_code,source_url,downloaded_utc,loaded_utc,rows_read,lines_repaired,lines_rejected,projects
0,1,HORIZON,https://cordis.europa.eu/data/cordis-HORIZONpr...,2026-11-16 20:47:09,2026-11-16 23:34:14,350442,82,0,24419
1,2,H2020,https://cordis.europa.eu/data/cordis-h2020proj...,2026-11-16 20:47:14,2026-11-16 23:34:14,659531,121,0,35389


## US12 - SME participation

In [14]:
run('us12_sme_participation')

-- US12. SME participation: share of participations declared by SMEs and their share of EU contribution, by programme.
SELECT programme_code,
       ROUND(100.0 * AVG(CASE WHEN is_sme THEN 1 ELSE 0 END), 1) AS sme_participation_pct,
       ROUND(100.0 * SUM(CASE WHEN is_sme THEN net_ec_contribution ELSE 0 END) / SUM(net_ec_contribution), 1) AS sme_funding_pct
FROM mart.fact_participation
GROUP BY programme_code ORDER BY programme_code;



,programme_code,sme_participation_pct,sme_funding_pct
0,H2020,19.8,16.8
1,HORIZON,17.7,16.1


## The same drill-down in SPARQL, on the ontology-aligned graph
The graph (`semantic/graph/horizon_ai_sample.ttl`) maps the model to SKOS, W3C ORG, FOAF, DCAT and Dublin Core. `skos:broader*` follows the EuroSciVoc tree at any depth: the result must match US10 (1,263 projects).

In [15]:
from rdflib import Graph
g = Graph().parse(ROOT / 'semantic' / 'graph' / 'horizon_ai_sample.ttl')
q = '''PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX dcterms: <http://purl.org/dc/terms/>
SELECT (COUNT(DISTINCT ?p) AS ?projects) WHERE {
  ?root skos:prefLabel "artificial intelligence"@en .
  ?d skos:broader* ?root . ?p dcterms:subject ?d }'''
print(len(g), 'triples'); [int(r[0]) for r in g.query(q)]

114836 triples


[1263]

## Limits
- Amounts are commitments; most Horizon Europe projects are still running.
- 18% of Horizon Europe projects have no EuroSciVoc domain in CORDIS.
- Coordination rates depend on the mix of single-beneficiary grants (ERC, MSCA, EIC) and on association rules, which are not analysed here.